In [ ]:
!pip install pdfplumber
!pip install tqdm
!pip install python-docx
!pip install textract

In [19]:
import os
import zipfile
import pandas as pd
import pdfplumber
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from tqdm.notebook import tqdm     
#import docx
from docx import Document 
import textract  

In [3]:
zip_file_path =r"D:\PYTHON\PATH\RFC-FS.zip"
output_dir = r'D:\PYTHON\Claude\PATH MODIFY'
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    print("✅ Extraction successful!")
except FileNotFoundError:
    print("Zip file not found.")
except Exception as e:
    print(f"Error: {e}")

✅ Extraction successful!


In [4]:
pdf_files = [
    os.path.join(root, name)
    for root, _, files in os.walk(output_dir)
    for name in files
    if name.lower().endswith('.pdf')
]
print(f"📄 Found {len(pdf_files)} PDF files")

📄 Found 7 PDF files


In [5]:
def read_pdf(pdf_path: str) -> dict:
    try:
        text = ""
        num_pages = 0
        with pdfplumber.open(pdf_path) as pdf:
            num_pages = len(pdf.pages)
            for page in pdf.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"

        return {
            "file_name": os.path.basename(pdf_path),
            "file_path": pdf_path,
            "num_pages": num_pages,
            "text": text.strip(),
            "status": "ok",
            "error": None
        }

    except Exception as e:
        return {
            "file_name": os.path.basename(pdf_path),
            "file_path": pdf_path,
            "num_pages": 0,
            "text": None,
            "status": "error",
            "error": str(e)
        }

In [6]:
records = []
MAX_WORKERS = 4  # you can tweak this

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(read_pdf, path) for path in pdf_files]

    for future in tqdm(as_completed(futures), total=len(pdf_files), desc="Reading PDFs"):
        records.append(future.result())

Reading PDFs: 100%|██████████████████████████████████████████████████████████████████████| 7/7 [00:05<00:00,  1.23it/s]


In [7]:
df = pd.DataFrame(records)
df.head()

,file_name,file_path,num_pages,text,status,error
0,HBL - RFC - 24101401 - MPSD Services R2P and R...,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - SMS ...,3,SOFTWARE CUSTOMIZATION CHANGE REQUEST FORM\nHB...,ok,None
1,HBL-RFC 24086601 - Bangla QR v.2.3.pdf,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - Bang...,4,SOFTWARE CUSTOMIZATION CHANGE REQUEST FORM\nHB...,ok,None
2,HBL_RFC 21118405_Visa 8 Digit mandate v.4.pdf,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - 8 di...,4,SOFTWARE CUSTOMIZATION CHANGE REQUEST FORM\nRF...,ok,None
3,HBL RFC 21028402 - MPSD Integration Phase-2 v....,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - Mald...,6,SOFTWARE CUSTOMIZATION CHANGE REQUEST FORM\nRF...,ok,None
4,HBL-RFC - 23058402-Masking in Unison v.1.3.pdf,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL Maskin...,4,SOFTWARE CUSTOMIZATION CHANGE REQUEST FORM\nRF...,ok,None


In [ ]:
#Totol doc file

In [8]:
doc_files = [
    os.path.join(root, name)
    for root, _, files in os.walk(output_dir)
    for name in files
    if name.lower().endswith(('.doc', '.docx'))
]
print(f"📄 Found {len(doc_files)} DOC/DOCX files")

📄 Found 13 DOC/DOCX files


In [9]:
docx_files = [
    os.path.join(root, name)
    for root, _, files in os.walk(output_dir)
    for name in files
    if name.lower().endswith('.docx')
]
print(f"📄 Found {len(docx_files)} .docx files")

📄 Found 10 .docx files


In [10]:
doc_files = [
    os.path.join(root, name)
    for root, _, files in os.walk(output_dir)
    for name in files
    if name.lower().endswith(('.doc'))
]
print(f"📄 Found {len(doc_files)} DOC files")

📄 Found 3 DOC files


In [20]:
def extract_text_from_docx(file_path):
    text = ''
    try:
        doc = Document(file_path)        
        for paragraph in doc.paragraphs:
            text += paragraph.text + '\n'
    except Exception as e:
        print(f"Error extracting text from DOCX {file_path}: {e}")
        text = f"ERROR: Could not extract text - {e}"
    return {
        'file_path': file_path,
        'file_name': os.path.basename(file_path),
        'extracted_text': text.strip()
    }

In [21]:
MAX_WORKERS = min(16, (os.cpu_count() or 1) * 2)
docx_records = []

print("Processing .docx files...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(extract_text_from_docx, path) for path in docx_files]

    for future in tqdm(as_completed(futures), total=len(futures)):
        docx_records.append(future.result())


df_docx = pd.DataFrame(docx_records)

print("Done!")
df_docx.head()

Processing .docx files...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 14.67it/s]

Done!


,file_path,file_name,extracted_text
0,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - HBL-...,FSD- Emirates ID Enablement on CCDM in UAE v1....,Emirates ID enablement on CCDM in UAE\nFunctio...
1,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - EStm...,HBL E-statement for New CC Customers on Mobile...,Document Information\n\n\n\nRevision History\n...
2,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - SMS ...,FSD HBL MPSD Notifications.docx,RFC for MPSD Notifications\nFunctional Specifi...
3,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL - HBL-...,HBL - RFC - 24011402 - Emirates ID on CCDM-2 3...,SOFTWARE CUSTOMIZATION CHANGE REQUEST FORM\nHB...
4,D:\PYTHON\Claude\PATH MODIFY\RFC-FS\HBL Maskin...,[HBL][FSD] - Data Masking 25Sep23.docx,Document Information\n\n\n\n\n\n\n\n\n\n\n\n\n...


In [44]:
def convert_doc_to_docx(doc_path: str) -> str:
        tmp_dir = tempfile.mkdtemp()
        subprocess.run(
            ["libreoffice", "--headless", "--convert-to", "docx",
             "--outdir", tmp_dir, doc_path],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True
        )
        base = os.path.splitext(os.path.basename(doc_path))[0]
        return os.path.join(tmp_dir, base + ".docx")

In [31]:
def extract_text_from_docx(file_path):
    text = ''
    try:
        doc = Document(file_path)        
        for paragraph in doc.paragraphs:
            text += paragraph.text + '\n'
    except Exception as e:
        print(f"Error extracting text from DOCX {file_path}: {e}")
        text = f"ERROR: Could not extract text - {e}"
    return {
        'file_path': file_path,
        'file_name': os.path.basename(file_path),
        'extracted_text': text.strip()
    }

In [45]:
  def process_file(path):
        # Convert .doc to .docx first
        if path.lower().endswith('.doc'):
            path = convert_doc_to_docx(path)
        return extract_text_from_docx(path)


In [42]:

MAX_WORKERS = min(16, (os.cpu_count() or 1) * 2)
docx_records = []

print("Processing .docx files...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(extract_text_from_docx, path) for path in docx_files]

    for future in tqdm(as_completed(futures), total=len(futures)):
        docx_records.append(future.result())


df_docx = pd.DataFrame(docx_records)

print("Done!")
df_docx.head()

Processing Word files...


  0%|                                                                                            | 0/3 [00:00<?, ?it/s]


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [41]:
import tempfile
import subprocess